# Read what the kernel certified about an AI proof

*Adversarial Mathematics From First Principles*, [chapter 6](https://programmer.ie/books/math/06-chapter/) · [worked example](https://programmer.ie/examples/math/06-chapter/)

Run the cells in order. Everything below the program is taken from the laboratory's recorded, hash-verified run; the program is a small, self-contained demonstration of the same mathematics.

A 2026 paper (Lau, arXiv:2605.20120v1) attaches a Lean artifact on IMO 2009
Problem 6, the grasshopper problem. The question is not whether the paper is
convincing but what Lean's kernel certified, declaration by declaration. The
audit compiled five variants of the artifact; this program takes what
`#print axioms` reported for each and re-derives the findings.

## Run it

Standard library only; nothing to install.

In [1]:
"""What Lean's kernel certified about the Grasshopper artifact, re-derived from its axioms.

The data below is what `#print axioms` reported for each declaration in the five
variants of the 2026-11-13 audit (Lean 4.28.0), plus the number of error lines in
each log. Declaration names and axiom lists only: the artifact itself is not
reproduced. Standard library only.
"""
STANDARD = {"propext", "Classical.choice", "Quot.sound"}
CLEAN = sorted(STANDARD)
SORRY = CLEAN + ["sorryAx"]
HELPERS = ["PS_last", "PS_swap", "PS_swap_eq", "maximizer_swap_in_M"]

RECORDED = {   # variant: (error lines, {declaration: axioms})
    "original": (0, {**{h: CLEAN for h in HELPERS}, "grasshopper": SORRY, "n3_bind_grasshopper": SORRY}),
    "without_main": (0, {h: CLEAN for h in HELPERS}),
    "hidden_sorry": (0, {"PS_last": CLEAN, "PS_swap": CLEAN, "PS_swap_eq": SORRY,
                         "maximizer_swap_in_M": SORRY, "grasshopper": SORRY}),
    "changed_statement": (2, {"PS_last": CLEAN, "PS_swap": CLEAN, "PS_swap_eq": SORRY,
                              "maximizer_swap_in_M": SORRY, "grasshopper": SORRY}),
    "substitution": (1, {**{h: CLEAN for h in HELPERS}, "grasshopper": CLEAN, "n3_bind_grasshopper": SORRY}),
}


def clean(axioms):
    return set(axioms) <= STANDARD


for variant, (errors, decls) in RECORDED.items():
    on_sorry = [d for d, ax in decls.items() if "sorryAx" in ax]
    print(f"{variant:18} errors {errors}  resting on sorryAx: {', '.join(on_sorry) or 'none'}")

o_err, o = RECORDED["original"]
h_err, h = RECORDED["hidden_sorry"]
c_err, _ = RECORDED["changed_statement"]
s_err, s = RECORDED["substitution"]
findings = {
    "original: the four helpers are axiom-clean": all(clean(o[x]) for x in HELPERS),
    "original: the main theorem rests on sorryAx": "sorryAx" in o["grasshopper"],
    "hidden gap reaches PS_swap_eq and maximizer_swap_in_M only":
        [x for x in HELPERS if not clean(h[x])] == ["PS_swap_eq", "maximizer_swap_in_M"],
    "changed helper statement is refused by the kernel": c_err > 0,
    "substituted main theorem is clean but does not bind": clean(s["grasshopper"]) and s_err > 0,
}
print()
for name, ok in findings.items():
    print(f"  {'yes' if ok else 'NO ':3}  {name}")
print()
# Failure control: the claim 'the artifact proves its main theorem' must be refused.
print(f"control: main theorem axiom-clean in the original? {clean(o['grasshopper'])} -> claim refused")

original           errors 0  resting on sorryAx: grasshopper, n3_bind_grasshopper
without_main       errors 0  resting on sorryAx: none
hidden_sorry       errors 0  resting on sorryAx: PS_swap_eq, maximizer_swap_in_M, grasshopper
changed_statement  errors 2  resting on sorryAx: PS_swap_eq, maximizer_swap_in_M, grasshopper
substitution       errors 1  resting on sorryAx: n3_bind_grasshopper

  yes  original: the four helpers are axiom-clean
  yes  original: the main theorem rests on sorryAx
  yes  hidden gap reaches PS_swap_eq and maximizer_swap_in_M only
  yes  changed helper statement is refused by the kernel
  yes  substituted main theorem is clean but does not bind

control: main theorem axiom-clean in the original? False -> claim refused


## What the output shows

The four helper lemmas rest only on Lean's standard axioms. The main theorem
rests on `sorryAx`: it is not proved. That is the result, not a defect of this
page. The other variants show the audit could see what it claims to see: an
injected gap surfaces exactly where it should, a changed statement is refused,
and a trivially restated main theorem is clean but does not bind to the
frozen statement. The artifact itself is not reproduced; its licence is
unknown.

## What the laboratory checked

The recorded run `20261113T232915Z-a62abc70` ([evidence](https://github.com/ernanhughes/math/tree/main/experiments/investigations/n3-grasshopper/runner-runs/2026-11-13-pinned)) is the book's result; the program above is a reader-sized demonstration of the same mathematics. **Evidence:** Verified historical Lean audit (imported, not re-run), freshly re-read from its kernel logs. **Status:** `historical_verified_correspondence_pending`.

> In the published artifact, the four helper lemmas rest only on Lean's standard axioms, while the main theorem rests on sorryAx; a gap injected into one helper surfaces in exactly the helpers that depend on it; a changed helper statement is refused; and a main theorem restated so that it proves trivially is axiom-clean but fails to bind to the frozen statement.

| Check | System | Recorded outcome |
| --- | --- | --- |
| `inputs` | Pinned inputs | all inputs match their pins |
| `audit-2026-11-13` | Imported record (not re-run) | 7 recorded files match the pinned manifest |
| `logs` | Python (independent computation) | exit 0, verdict main_theorem_rests_on_sorry |

Failure controls: **1 of 1** rejected or refused as required (`claim-main-proved`).
Correspondence `N3-CORR-01` (does the formal statement say what the claim means?): **pending the author's approval**.
Contract `N3-GRASSHOPPER` version 1, sha256 `c19401797afe6fdd…`; `result.json` sha256 `c9bb9c27d5f152a3…`.

## Not shown

As the laboratory's own record states:

- A fresh kernel execution: this run imports the 2026-11-13 kernel output and re-reads it. Re-running the kernel means the separate grasshopper-env procedure.
- That the artifact's main statement is IMO 2009 Problem 6: that comparison awaits human review.
- Anything about what the AI system contributed to the artifact, or about the paper's claims.
- Permission to publish the artifact or its logs: the licence is unknown, and the logs quote elaborated statements.

## Reproduce or challenge it

To reproduce the laboratory's run, build the `math-env` image from a clone of the repository ([Getting Started](/books/math/29-chapter/)) and run:

```bash
img=$(docker image inspect math-env --format '{{.Id}}')
docker run --rm --network none -e UV_OFFLINE=1 -e MATHENV_IMAGE_ID=$img -v "${PWD}:/source:ro" -v "${PWD}/out:/out" math-env uv run --offline math-engine proof grasshopper --out /out/proof-grasshopper
```

This Docker form was tested for `fermat`, `kempe` and `simpson`, not separately for `grasshopper`. To re-verify the recorded run itself without Docker, `math-engine proof --verify experiments/investigations/n3-grasshopper/runner-runs/2026-11-13-pinned` (R-V05).

The runner re-verifies the imported audit and re-reads its kernel logs; it does **not** re-run Lean on the artifact. A fresh audit needs the separate Lean 4.28.0 image (`docker build -t grasshopper-env:4.28.0 docker/grasshopper`, then `audit.py`).

Examine the audit's conclusion that the main theorem rests on `sorryAx` while
the four helper lemmas are axiom-clean. Could a declaration be axiom-clean and
still not support the paper's claim? What would you need to check before
calling the artifact a proof of IMO 2009 Problem 6? Keep separate the imported
kernel record and the fresh reading of it, and do not reproduce the
artifact's code.